## Setup

In [1]:
import anndata as ad
import crested
import numpy as np
import matplotlib

# Set the font type to ensure text is saved as whole words
matplotlib.rcParams["pdf.fonttype"] = 42  # Use TrueType fonts instead of Type 3 fonts
matplotlib.rcParams["ps.fonttype"] = 42  # For PostScript as well, if needed


## Prepare data

To train a CREsted peak regression model on your data, we need:

    1. A consensus regions BED file containing all the regions of interest accross cell types.

    2. A folder containing the bigwig files per cell type. Each file should be named according to the cell type: {cell type name}.bw.

    3. A genome fasta file and optionally a chromosome sizes file.



In [ ]:
# Load genome
genome_file = "genome/Nvec_vc1.1_gDNA.fasta"
chrom_sizes = "genome/Nvec_vc1.1_gDNA.chromsizes"
genome = crested.Genome(genome_file, chrom_sizes)

# Register the genome so that it can be used by the package
crested.register_genome(genome)

2025-02-25T19:11:24.751987+0100 INFO Genome Nvec_vc1.1_gDNA registered.


In [ ]:
# Consensus regions BED file
regions_file = "results/CREsted/Peaks_cell_type.bed"

# Folder with bigwig files
bigwigs_folder = "results/CREsted/bw"

# Import the data
adata = crested.import_bigwigs(
    bigwigs_folder=bigwigs_folder,
    regions_file=regions_file,
    target_region_width=250,  # optionally, use a different width than the consensus regions file (500bp) for the .X values calculation
    target="logcount"  # or "max", "mean", "logcount" --> what we will be predicting
)
adata

2025-02-24T13:03:13.785736+0100 INFO Extracting values from 22 bigWig files...


AnnData object with n_obs × n_vars = 22 × 112671
    obs: 'file_path'
    var: 'chr', 'start', 'end'

Train - validation - test split

In [4]:
# Choose the chromosomes for the validation and test sets
crested.pp.train_val_test_split(
    adata, strategy="chr", val_chroms=["NC_064034.1"], test_chroms=["NC_064035.1"]
)

print(adata.var["split"].value_counts())

split
train    92697
test     10085
val       9889
Name: count, dtype: int64


Preprocessing

In [5]:
# Change region width
crested.pp.change_regions_width(adata, 500)

2025-02-24T13:03:42.602019+0100 WARNING Region NC_064041.1:-13-487 with coordinates NC_064041.1:-13-487 is out of bounds for chromosome NC_064041.1. Removing region.
2025-02-24T13:03:43.418571+0100 WARNING Region NC_064048.1:13716980-13717480 with coordinates NC_064048.1:13716980-13717480 is out of bounds for chromosome NC_064048.1. Removing region.


In [6]:
# Substract mean peak value
adata.X = adata.X - np.mean(adata.X, axis=0)

In [7]:
# Save the final preprocessing results
adata.write_h5ad("data/nematostella_cell_type_500region_250target_logcount_meansubtract.h5ad")

## Train the model

We’ll start by initializing the crested.tl.data.AnnDataModule object with our data.  
This will tell our model how to load the data and what data to load during fitting/evaluation. 

In [8]:
import keras
print(keras.backend.backend())

torch


In [ ]:
# read in your preprocessed data
adata = ad.read_h5ad("results/CREsted/nematostella_gastrula_500region_250target_logcount_meansubtract.h5ad")

datamodule = crested.tl.data.AnnDataModule(
    adata,
    batch_size=256,  # lower this if you encounter OOM errors
    max_stochastic_shift=3,  # optional data augmentation to slightly reduce overfitting
    always_reverse_complement=True,  # default True. Will double the effective size of the training dataset.
)

# Load CNN architecture for a dataset with 500bp regions and cell types as classes
model_architecture = crested.tl.zoo.deeptopic_cnn(
    seq_len=500, num_classes=len(list(adata.obs_names)), output_activation="linear"
)

# Load the default configuration for training a peak regression model
#config = crested.tl.default_configs("peak_regression") 
#print(config)

import keras

# Create your own configuration for training a peak regression model
optimizer = keras.optimizers.Adam(learning_rate=5e-4)
loss = keras.losses.Huber(delta=1.0)
metrics = [
    keras.metrics.MeanAbsoluteError(),
    keras.metrics.MeanSquaredError(),
    keras.metrics.CosineSimilarity(axis=1),
    crested.tl.metrics.PearsonCorrelation(),
    crested.tl.metrics.ConcordanceCorrelationCoefficient(),
    crested.tl.metrics.PearsonCorrelationLog(),
    crested.tl.metrics.ZeroPenaltyMetric(),
]

config = crested.tl.TaskConfig(optimizer, loss, metrics)
print(config)

# Setup the trainer
trainer = crested.tl.Crested(
    data=datamodule,
    model=model_architecture,
    config=config,
    project_name="models",
    run_name="meansubstract_deeptopic_gastrula_500_logcount_huber",
    logger="None",
    seed=1950,
)

# Train the model
trainer.fit(
    epochs=60,
    learning_rate_reduce_patience=3,
    early_stopping_patience=6,
)

TaskConfig(optimizer=<keras.src.backend.torch.optimizers.torch_adam.Adam object at 0x7f517d1a0680>, loss=<LossFunctionWrapper(<function huber at 0x7f5568ba5d00>, kwargs={'delta': 1.0})>, metrics=[<MeanAbsoluteError name=mean_absolute_error>, <MeanSquaredError name=mean_squared_error>, <CosineSimilarity name=cosine_similarity>, <PearsonCorrelation name=pearson_correlation>, <ConcordanceCorrelationCoefficient name=concordance_correlation_coefficient>, <PearsonCorrelationLog name=pearson_correlation_log>, <ZeroPenaltyMetric name=zero_penalty_metric>])
2025-02-24T12:00:12.291904+0100 WARNING Output directory models/meansubstract_deeptopic_gastrula_500_logcount_huber/checkpoints, already exists but no trained models found. Overwriting...


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 500, 4)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 500, 1024) │     69,632 │ sequence[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 500, 1024) │      4,096 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_5        │ (None, 500, 1024) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_4     │ (None, 125, 1024) │          0 │ activation_5[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 125, 1024) │          0 │ max_pooling1d_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 125, 512)  │  5,767,168 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 125, 512)  │      2,048 │ conv1d_6[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 125, 512)  │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_5     │ (None, 32, 512)   │          0 │ activation_6[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 32, 512)   │          0 │ max_pooling1d_5[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_7 (Conv1D)   │ (None, 32, 512)   │  2,883,584 │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 512)   │      2,048 │ conv1d_7[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_7        │ (None, 32, 512)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_6     │ (None, 8, 512)    │          0 │ activation_7[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 8, 512)    │          0 │ max_pooling1d_6[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 8, 512)    │  1,310,720 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 512)    │      2,048 │ conv1d_8[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_8        │ (None, 8, 512)    │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                 

 Total params: 11,630,602 (44.37 MB)

 Trainable params: 11,622,410 (44.34 MB)

...

None
2025-02-24T12:00:12.438630+0100 INFO Loading sequences into memory...


100%|██████████| 92695/92695 [00:00<00:00, 112096.24it/s]


2025-02-24T12:00:13.420083+0100 INFO Loading sequences into memory...


100%|██████████| 9889/9889 [00:00<00:00, 225432.07it/s]


Epoch 1/60
725/725 ━━━━━━━━━━━━━━━━━━━━ 127s 175ms/step - concordance_correlation_coefficient: 0.1740 - cosine_similarity: 0.2902 - loss: 0.3020 - mean_absolute_error: 0.5975 - mean_squared_error: 0.6175 - pearson_correlation: 0.3001 - pearson_correlation_log: nan - zero_penalty_metric: 88.3762 - val_concordance_correlation_coefficient: 0.2404 - val_cosine_similarity: 0.3115 - val_loss: 0.2824 - val_mean_absolute_error: 0.5982 - val_mean_squared_error: 0.6184 - val_pearson_correlation: 0.3272 - val_pearson_correlation_log: nan - val_zero_penalty_metric: 75.6508 - learning_rate: 5.0000e-04
Epoch 2/60
725/725 ━━━━━━━━━━━━━━━━━━━━ 127s 175ms/step - concordance_correlation_coefficient: 0.2153 - cosine_similarity: 0.3194 - loss: 0.2748 - mean_absolute_error: 0.5893 - mean_squared_error: 0.6023 - pearson_correlation: 0.3366 - pearson_correlation_log: nan - zero_penalty_metric: 90.8378 - val_concordance_correlation_coefficient: 0.2542 - val_cosine_similarity: 0.3272 - val_loss: 0.2763 - val_m

## Finetuning on cell type-specific regions

We continue training the model trained on all consensuspeaks on a subset of cell type-specific regions. We define specific regions as regions with a high Gini index, indicating that their peak distribution over all cell types will be skewed and specific for one or more cell types.

In [ ]:
import anndata as ad
import crested

# read in your preprocessed data
adata = ad.read_h5ad("results/CREsted/nematostella_broadcelltype_500region_250target_logcount_meansubtract.h5ad")
adata

AnnData object with n_obs × n_vars = 17 × 89861
    obs: 'file_path'
    var: 'chr', 'start', 'end', 'split'

In [4]:
# Keep regions with a Gini index 1 std above the mean across all regions
crested.pp.filter_regions_on_specificity(
    adata, gini_std_threshold=0.5
)
adata

2025-02-25T19:12:00.136242+0100 INFO After specificity filtering, kept 27542 out of 89861 regions.


AnnData object with n_obs × n_vars = 17 × 27542
    obs: 'file_path'
    var: 'chr', 'start', 'end', 'split'

In [ ]:
# Save filtered file
adata.write_h5ad("results/CREsted/nematostella_broadcelltype_500region_250target_logcount_meansubtract_filtered.h5ad")

Loading the pretrained model on all consensuspeaks and finetuning with lower learning rate

In [ ]:
# Load genome
genome_file = "genome/Nvec_vc1.1_gDNA.fasta"
chrom_sizes = "genome/Nvec_vc1.1_gDNA.chromsizes"
genome = crested.Genome(genome_file, chrom_sizes)

# Register the genome so that it can be used by the package
crested.register_genome(genome)

# Datamodule with filtered data
datamodule = crested.tl.data.AnnDataModule(
    adata,
    batch_size=64,  # Recommended to go for a smaller batch size than in the basemodel
    max_stochastic_shift=3,
    always_reverse_complement=True,
)

import keras

# First load the pretrained model on all peaks
model_architecture = keras.models.load_model(
    "results/CREsted/models/meansubstract_deeptopic_broadcelltype_500_logcount_huber.keras",
    compile=False,  # Choose the basemodel with best validation loss/performance metrics
)

# Use the same config you used for the pretrained model. 
# EXCEPT THE LEARNING RATE, make sure that is lower than it was on the epoch you select the model from
optimizer = keras.optimizers.Adam(learning_rate=1e-5)
loss = keras.losses.Huber(delta=1.0)
metrics = [
    keras.metrics.MeanAbsoluteError(),
    keras.metrics.MeanSquaredError(),
    keras.metrics.CosineSimilarity(axis=1),
    crested.tl.metrics.PearsonCorrelation(),
    crested.tl.metrics.ConcordanceCorrelationCoefficient(),
    crested.tl.metrics.PearsonCorrelationLog(),
    crested.tl.metrics.ZeroPenaltyMetric(),
]

config = crested.tl.TaskConfig(optimizer, loss, metrics)
print(config)

# Setup the trainer
trainer = crested.tl.Crested(
    data=datamodule,
    model=model_architecture,
    config=config,
    project_name="model",
    run_name="meansubstract_deeptopic_broadcelltype_500_logcount_huber_finetuned",
    logger="None",
    seed=1950,
)

# Train the finetuned model
trainer.fit(
    epochs=40,
    learning_rate_reduce_patience=3,
    early_stopping_patience=6,
)

2025-02-25T19:12:43.882933+0100 INFO Genome Nvec_vc1.1_gDNA registered.
TaskConfig(optimizer=<keras.src.backend.torch.optimizers.torch_adam.Adam object at 0x7fc5cb0c88f0>, loss=<LossFunctionWrapper(<function huber at 0x7fc932b65d00>, kwargs={'delta': 1.0})>, metrics=[<MeanAbsoluteError name=mean_absolute_error>, <MeanSquaredError name=mean_squared_error>, <CosineSimilarity name=cosine_similarity>, <PearsonCorrelation name=pearson_correlation>, <ConcordanceCorrelationCoefficient name=concordance_correlation_coefficient>, <PearsonCorrelationLog name=pearson_correlation_log>, <ZeroPenaltyMetric name=zero_penalty_metric>])


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 500, 4)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 500, 1024) │     69,632 │ sequence[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 500, 1024) │      4,096 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 500, 1024) │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 125, 1024) │          0 │ activation[0][0]  │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 125, 1024) │          0 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 125, 512)  │  5,767,168 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 125, 512)  │      2,048 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 125, 512)  │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 32, 512)   │          0 │ activation_1[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32, 512)   │          0 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 32, 512)   │  2,883,584 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 512)   │      2,048 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 32, 512)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 8, 512)    │          0 │ activation_2[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 8, 512)    │          0 │ max_pooling1d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 8, 512)    │  1,310,720 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 512)    │      2,048 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 8, 512)    │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                 

 Total params: 11,637,777 (44.39 MB)

 Trainable params: 11,629,585 (44.36 MB)

...

2025-02-25T19:12:44.436378+0100 INFO Loading sequences into memory...


100%|██████████| 1361/1361 [00:00<00:00, 173942.58it/s]

Epoch 1/40



/users/asebe/aelek/bin/miniconda3/lib/python3.12/site-packages/keras/src/backend/torch/nn.py:466: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1036.)
  outputs = tnn.conv1d(


774/774 ━━━━━━━━━━━━━━━━━━━━ 46s 59ms/step - concordance_correlation_coefficient: 0.3204 - cosine_similarity: 0.3783 - loss: 0.2907 - mean_absolute_error: 0.5748 - mean_squared_error: 0.5902 - pearson_correlation: 0.4541 - pearson_correlation_log: nan - zero_penalty_metric: 0.0000e+00 - val_concordance_correlation_coefficient: 0.3442 - val_cosine_similarity: 0.3580 - val_loss: 0.2565 - val_mean_absolute_error: 0.5249 - val_mean_squared_error: 0.5068 - val_pearson_correlation: 0.4605 - val_pearson_correlation_log: nan - val_zero_penalty_metric: 0.0000e+00 - learning_rate: 1.0000e-05
Epoch 2/40
774/774 ━━━━━━━━━━━━━━━━━━━━ 45s 58ms/step - concordance_correlation_coefficient: 0.3463 - cosine_similarity: 0.3932 - loss: 0.2856 - mean_absolute_error: 0.5685 - mean_squared_error: 0.5773 - pearson_correlation: 0.4729 - pearson_correlation_log: nan - zero_penalty_metric: 0.0000e+00 - val_concordance_correlation_coefficient: 0.3576 - val_cosine_similarity: 0.3642 - val_loss: 0.2549 - val_mean_ab